# CIFAR-10 Control Study

Self-contained. Rebuilds the CIFAR-10 baseline (the original `stages_*.pt` is gone)
and adds the analyses the first run never had: null baselines, per-victim floors,
and identifiability.

**Run order:** `0` -> `C1` -> `C2` -> `C3` -> `C4` -> `C5`.
Use `R` instead of `C1` when resuming after a session drop.

Set `SMOKE = True` in Cell 0, re-run Cell 0, run `S`, then set it back to `False`
and re-run Cell 0 before `C1`.

**Download `outputs_cifar` after every cell.** `C3` checkpoints after each condition
and resumes on re-run, so an interruption costs one condition, not the whole cell.


In [ ]:
# =============================================================================
# CELL 0 - LIBRARY.  Run this first, every session.  Defines everything; runs nothing.
#
# Self-contained CIFAR-10 control study. Rebuilds the baseline from scratch (the
# original stages_*.pt is gone) and adds the analyses the first CIFAR run never had:
# null baselines, per-victim floors, and identifiability. Config is copied exactly
# from the original notebook so the baseline should land near 46.0% accuracy.
# =============================================================================
import os, io, sys, json, math, copy, time, random, shutil
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats as _scistats
from skimage.metrics import peak_signal_noise_ratio as _psnr
from skimage.metrics import structural_similarity as _ssim

SMOKE = False        # True -> tiny everything, ~3 minute end-to-end check
ALLOW_SMOKE_CELLS = False


@dataclass
class Config:
    num_clients: int = 10
    alpha: float = 0.5
    rounds: int = 60
    local_epochs: int = 2
    local_bs: int = 64
    lr: float = 0.1
    num_classes: int = 10
    img: int = 32
    in_ch: int = 3
    data_root: str = "/kaggle/working/data"
    out_dir: str = "/kaggle/working/outputs_cifar"
    seed: int = 42
    dlg_iters: int = 300
    ig_iters: int = 1200
    restarts: int = 2
    n_victims: int = 10


cfg = Config()
if SMOKE:
    cfg.rounds, cfg.n_victims = 2, 3
    cfg.dlg_iters, cfg.ig_iters, cfg.restarts = 20, 40, 1
os.makedirs(cfg.out_dir, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS = {}


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def save_results(tag=""):
    p = os.path.join(cfg.out_dir, "results_cifar.json")
    with open(p, "w") as f:
        json.dump({"config": asdict(cfg), "results": RESULTS}, f, indent=2, default=float)
    print(f"[saved] {p} {tag}")


def mean_ci(vals, conf=0.95):
    a = np.asarray(vals, dtype=float); a = a[np.isfinite(a)]; n = len(a)
    if n == 0:
        return float("nan"), float("nan")
    m = float(a.mean())
    if n < 2:
        return m, 0.0
    se = float(a.std(ddof=1)) / math.sqrt(n)
    return m, float(se * _scistats.t.ppf(0.5 + conf / 2, n - 1))


def fmt(m, h):
    return f"{m:.3f} +/- {h:.3f}"


# ------------------------------- MODEL --------------------------------------
class LeNet(nn.Module):
    """Exactly the original: sigmoid activations, 12*8*8 = 768 flatten."""
    def __init__(self, num_classes=10):
        super().__init__()
        act = nn.Sigmoid
        self.body = nn.Sequential(
            nn.Conv2d(3, 12, 5, padding=2, stride=2), act(),
            nn.Conv2d(12, 12, 5, padding=2, stride=2), act(),
            nn.Conv2d(12, 12, 5, padding=2, stride=1), act())
        self.fc = nn.Linear(12 * 8 * 8, num_classes)

    def forward(self, x):
        return self.fc(self.body(x).view(x.size(0), -1))


def build_model(device=None):
    return LeNet(cfg.num_classes).to(device or DEVICE)


# -------------------------------- DATA --------------------------------------
def load_data(c):
    tf = T.Compose([T.ToTensor()])          # pixels in [0,1]; no normalization
    tr = torchvision.datasets.CIFAR10(c.data_root, train=True, download=True, transform=tf)
    te = torchvision.datasets.CIFAR10(c.data_root, train=False, download=True, transform=tf)
    print(f"cifar10: train {len(tr)}, test {len(te)}, {c.num_classes} classes")
    return tr, te


def dirichlet_partition(labels, num_clients, alpha, num_classes, seed=42):
    rng = np.random.default_rng(seed)
    labels = np.array(labels)
    client_idx = [[] for _ in range(num_clients)]
    for c in range(num_classes):
        idx_c = np.where(labels == c)[0]
        rng.shuffle(idx_c)
        props = rng.dirichlet(alpha * np.ones(num_clients))
        cut = (np.cumsum(props) * len(idx_c)).astype(int)[:-1]
        for i, part in enumerate(np.split(idx_c, cut)):
            client_idx[i].extend(part.tolist())
    for i in range(num_clients):
        rng.shuffle(client_idx[i])
    return client_idx


def make_client_loaders(c, train):
    parts = dirichlet_partition(train.targets, c.num_clients, c.alpha, c.num_classes, c.seed)
    if SMOKE:
        parts = [p[:300] for p in parts[:4]]
    return [DataLoader(Subset(train, i), batch_size=c.local_bs, shuffle=True)
            for i in parts], parts


# ------------------------------- FEDAVG -------------------------------------
def local_train(global_state, loader, c, device):
    m = build_model(device); m.load_state_dict(global_state); m.train()
    opt = torch.optim.SGD(m.parameters(), lr=c.lr)
    crit = nn.CrossEntropyLoss()
    for _ in range(c.local_epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); crit(m(x), y).backward(); opt.step()
    return m.state_dict()


def average_weights(states, sizes):
    total = float(sum(sizes)); avg = copy.deepcopy(states[0])
    for k in avg.keys():
        avg[k] = sum(s[k].float() * (n / total) for s, n in zip(states, sizes))
    return avg


@torch.no_grad()
def evaluate(state, loader, device):
    m = build_model(device); m.load_state_dict(state); m.eval()
    ok = tot = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        ok += (m(x).argmax(1) == y).sum().item(); tot += y.size(0)
    return ok / tot


def run_federated(c, loaders, test_loader, device, every=6):
    set_seed(c.seed)
    gs = build_model(device).state_dict()
    sizes = [len(l.dataset) for l in loaders]
    hist = []; t0 = time.time()
    for r in range(1, c.rounds + 1):
        gs = average_weights([local_train(gs, l, c, device) for l in loaders], sizes)
        if r % every == 0 or r == c.rounds:
            acc = evaluate(gs, test_loader, device)
            hist.append((r, acc))
            print(f"  [alpha={c.alpha}] round {r:3d}/{c.rounds}  acc {acc:.4f}  "
                  f"({time.time() - t0:.0f}s)")
    return gs, hist


def capture_gradient(state, x, y, device):
    m = build_model(device); m.load_state_dict(state); m.eval(); m.zero_grad()
    loss = nn.CrossEntropyLoss()(m(x.to(device)), y.to(device))
    return [g.detach().clone() for g in torch.autograd.grad(loss, m.parameters())]


# ------------------------------ METRICS -------------------------------------
def compute_metrics(recon, true):
    r = recon.detach().clamp(0, 1).squeeze(0).permute(1, 2, 0).cpu().numpy()
    t = true.detach().clamp(0, 1).squeeze(0).permute(1, 2, 0).cpu().numpy()
    try:
        s = _ssim(t, r, data_range=1.0, channel_axis=2)
    except TypeError:
        s = _ssim(t, r, data_range=1.0, multichannel=True)
    return float(_psnr(t, r, data_range=1.0)), float(s)


def total_variation(x):
    return ((x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean()
            + (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean())


# ------------------------------ DEFENSES ------------------------------------
def prune_gradient(grad, ratio):
    if not ratio:
        return grad
    flat = torch.cat([g.reshape(-1).abs() for g in grad])
    k = int(len(flat) * ratio)
    if k <= 0:
        return grad
    thr = torch.kthvalue(flat, k).values
    return [g * (g.abs() > thr) for g in grad]


def dp_gradient(grad, clip_C, sigma):
    scale = 1.0
    if clip_C is not None:
        norm = torch.sqrt(sum((g ** 2).sum() for g in grad))
        scale = min(1.0, float(clip_C) / float(norm + 1e-12))
    out = []
    for g in grad:
        gg = g * scale
        if sigma and sigma > 0:
            gg = gg + torch.randn_like(gg) * sigma
        out.append(gg)
    return out


def apply_defense(grad, spec):
    g = grad
    if spec.get("prune", 0) > 0:
        g = prune_gradient(g, spec["prune"])
    if spec.get("clip") is not None or spec.get("sigma", 0) > 0:
        g = dp_gradient(g, spec.get("clip"), spec.get("sigma", 0.0))
    return g


DEFENSES = [
    ("none",         {}),
    ("clip C=4",     {"clip": 4}),
    ("noise s=0.01", {"sigma": 0.01}),
    ("noise s=0.1",  {"sigma": 0.1}),
    ("prune 90%",    {"prune": 0.90}),
    ("combined",     {"prune": 0.90, "clip": 8, "sigma": 0.01}),
]


# ------------------------------- ATTACKS ------------------------------------
def _init_latent(device, seed):
    """Dedicated generator: the starting latent depends on the victim only, never on
    the defense, the attack, or execution history. The original notebook drew from the
    global RNG, which is why its IG moved 0.476 -> 0.498 under a pure rescaling."""
    g = torch.Generator(device="cpu").manual_seed(int(seed))
    return torch.randn(1, 3, cfg.img, cfg.img, generator=g).to(device).requires_grad_(True)


def dlg_attack(state, real_grad, label, device, iters=None, lr=1.0,
               restarts=None, seed_off=0):
    iters = cfg.dlg_iters if iters is None else iters
    restarts = cfg.restarts if restarts is None else restarts
    model = build_model(device); model.load_state_dict(state); model.eval()
    for p in model.parameters():
        p.requires_grad_(True)
    real_grad = [g.detach().to(device) for g in real_grad]
    label_t = torch.tensor([label], device=device)
    crit = nn.CrossEntropyLoss()
    bL, bI = float("inf"), None
    for rs in range(restarts):
        latent = _init_latent(device, cfg.seed + 1_000_003 * seed_off + 7919 * rs)

        def gm():
            dx = torch.sigmoid(latent)
            loss = crit(model(dx), label_t)
            dg = torch.autograd.grad(loss, model.parameters(), create_graph=True)
            return sum(((a - b) ** 2).sum() for a, b in zip(dg, real_grad))

        opt = torch.optim.LBFGS([latent], lr=lr, line_search_fn="strong_wolfe")
        l, im = float("inf"), torch.sigmoid(latent).detach().clone()
        for _ in range(iters):
            def closure():
                opt.zero_grad(); d = gm(); d.backward(); return d
            v = float(opt.step(closure).item())
            if np.isfinite(v) and v < l:
                l, im = v, torch.sigmoid(latent).detach().clone()
            if not np.isfinite(v):
                break
        if l < bL:
            bL, bI = l, im
    return bI, bL


def ig_attack(state, real_grad, label, device, iters=None, lr=0.1, tv_weight=1e-2,
              restarts=None, seed_off=0):
    iters = cfg.ig_iters if iters is None else iters
    restarts = cfg.restarts if restarts is None else restarts
    model = build_model(device); model.load_state_dict(state); model.eval()
    for p in model.parameters():
        p.requires_grad_(True)
    real_grad = [g.detach().to(device) for g in real_grad]
    rn = torch.sqrt(sum((g ** 2).sum() for g in real_grad))
    label_t = torch.tensor([label], device=device)
    crit = nn.CrossEntropyLoss()
    bL, bI = float("inf"), None
    for rs in range(restarts):
        latent = _init_latent(device, cfg.seed + 1_000_003 * seed_off + 7919 * rs)
        opt = torch.optim.Adam([latent], lr=lr)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=iters)
        l, im = float("inf"), torch.sigmoid(latent).detach().clone()
        for _ in range(iters):
            opt.zero_grad()
            dx = torch.sigmoid(latent)
            loss = crit(model(dx), label_t)
            dg = torch.autograd.grad(loss, model.parameters(), create_graph=True)
            dot = sum((a * b).sum() for a, b in zip(dg, real_grad))
            dn = torch.sqrt(sum((a ** 2).sum() for a in dg))
            cos = 1 - dot / (dn * rn + 1e-8)
            (cos + tv_weight * total_variation(dx)).backward()
            opt.step(); sched.step()
            v = float(cos.detach().item())
            if np.isfinite(v) and v < l:
                l, im = v, torch.sigmoid(latent).detach().clone()
        if l < bL:
            bL, bI = l, im
    return bI, bL


ATTACKS = {"DLG": dlg_attack, "IG": ig_attack}


# ------------------------------- FIGURES ------------------------------------
def _finish(fig, path):
    plt.close(fig)
    print(f"[fig] {path}")


def save_grid(images, titles, path, nrow=6, suptitle=None):
    n = len(images); ncol = int(math.ceil(n / nrow))
    fig, axes = plt.subplots(ncol, nrow, figsize=(1.6 * nrow, 1.85 * ncol), squeeze=False)
    for i, ax in enumerate(axes.ravel()):
        ax.set_xticks([]); ax.set_yticks([])
        if i < n:
            ax.imshow(images[i].detach().cpu().clamp(0, 1).squeeze(0).permute(1, 2, 0).numpy()
                      if images[i].dim() == 4 else
                      images[i].detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy())
            ax.set_title(titles[i], fontsize=8)
        else:
            ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.94 if suptitle else 1])
    fig.savefig(path, dpi=170)
    _finish(fig, path)


print("library ready | SMOKE =", SMOKE, "| device:", DEVICE,
      "| torch", torch.__version__)


In [ ]:
# =============================================================================
# CELL S - SMOKE TEST.  Set SMOKE = True in Cell 0, RE-RUN Cell 0, then run this.
# ~3 minutes. Exercises the whole pipeline on a tiny configuration so a typo costs
# minutes instead of an hour. Set SMOKE = False and re-run Cell 0 afterwards.
# =============================================================================
assert SMOKE, "Set SMOKE = True in Cell 0 and re-run Cell 0 before this cell."
_tr, _te = load_data(cfg)
_ld, _parts = make_client_loaders(cfg, _tr)
print("client sizes:", [len(l.dataset) for l in _ld])
_tl = DataLoader(Subset(_te, list(range(1000))), batch_size=256)
_gs, _h = run_federated(cfg, _ld, _tl, DEVICE, every=1)
_x = torch.stack([_te[i][0] for i in range(2)])
_y = torch.tensor([_te[i][1] for i in range(2)])
for k in range(2):
    _g = capture_gradient(_gs, _x[k:k + 1], _y[k:k + 1], DEVICE)
    for _an, _af in ATTACKS.items():
        _r, _l = _af(_gs, _g, int(_y[k]), DEVICE, seed_off=k)
        _p, _s = compute_metrics(_r, _x[k])
        print(f"  {_an} v{k}: psnr {_p:.2f} ssim {_s:.3f} loss {_l:.3e}")
_g = capture_gradient(_gs, _x[0:1], _y[0:1], DEVICE)
_a, _ = ATTACKS["IG"](_gs, _g, int(_y[0]), DEVICE, seed_off=0)
_b, _ = ATTACKS["IG"](_gs, apply_defense(_g, {"clip": 4}), int(_y[0]), DEVICE, seed_off=0)
print(f"  IG clip-invariance check: max abs diff {float((_a - _b).abs().max()):.2e} "
      f"(should be ~0 if the clip bound; check the gradient norm below)")
print(f"  gradient norm {float(torch.sqrt(sum((g ** 2).sum() for g in _g))):.3f} "
      f"(clip C=4 binds only if this exceeds 4)")
print("SMOKE PASSED")


In [ ]:
# =============================================================================
# CELL C1 - BASELINE.  FedAvg on CIFAR-10, 10 clients, Dirichlet alpha=0.5, 60 rounds.
# ~18 min. Config copied from the original notebook, so this should land near 46.0%.
# Saves the trained state so no later cell ever needs to retrain.
# =============================================================================
assert not SMOKE or ALLOW_SMOKE_CELLS, \
    "SMOKE is still True. Set SMOKE = False in Cell 0 and re-run Cell 0 first."

train, test = load_data(cfg)
loaders, parts = make_client_loaders(cfg, train)
test_loader = DataLoader(test, batch_size=256)
print("client sizes:", [len(l.dataset) for l in loaders])
print("params:", sum(p.numel() for p in build_model().parameters()))

gstate, hist = run_federated(cfg, loaders, test_loader, DEVICE, every=6)
final_acc = hist[-1][1]
print(f"\nCIFAR-10 alpha={cfg.alpha}: final acc {final_acc:.4f}   "
      f"(original notebook reported 0.460)")

set_seed(cfg.seed)
victims = sorted(int(i) for i in
                 np.random.default_rng(cfg.seed).choice(len(test), cfg.n_victims, False))
VX = torch.stack([test[i][0] for i in victims])
VY = torch.tensor([int(test[i][1]) for i in victims])
print("victims:", victims, "| labels:", VY.tolist())

RESULTS["baseline"] = {"dataset": "cifar10", "alpha": cfg.alpha, "rounds": cfg.rounds,
                       "final_acc": final_acc, "acc_history": hist,
                       "params": sum(p.numel() for p in build_model().parameters()),
                       "client_sizes": [len(l.dataset) for l in loaders],
                       "victims": victims, "victim_labels": VY.tolist()}
save_results("C1")
torch.save({"gstate": gstate, "victims": victims, "hist": hist},
           os.path.join(cfg.out_dir, "cifar_baseline.pt"))
print("[saved]", os.path.join(cfg.out_dir, "cifar_baseline.pt"))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot([r for r, _ in hist], [a for _, a in hist], marker="o")
ax.set_xlabel("round"); ax.set_ylabel("test accuracy"); ax.grid(alpha=.3)
ax.set_title(f"CIFAR-10, {cfg.num_clients} clients, Dirichlet alpha={cfg.alpha}")
fig.tight_layout()
_p = os.path.join(cfg.out_dir, "cifar_acc.png"); fig.savefig(_p, dpi=160); _finish(fig, _p)
save_grid([VX[k] for k in range(len(VX))], [f"v{k} c{int(VY[k])}" for k in range(len(VX))],
          os.path.join(cfg.out_dir, "cifar_victims.png"), nrow=10, suptitle="CIFAR-10 victims")


In [ ]:
# =============================================================================
# CELL R - RESTORE.  Run INSTEAD of C1 when resuming after a session drop.
# Loads the trained model from the working directory or an attached Kaggle dataset.
# =============================================================================
def _find(fname):
    cands = [os.path.join(cfg.out_dir, fname)]
    if os.path.isdir("/kaggle/input"):
        for d in os.listdir("/kaggle/input"):
            cands.append(os.path.join("/kaggle/input", d, fname))
    return next((p for p in cands if os.path.exists(p)), None)


os.makedirs(cfg.out_dir, exist_ok=True)
_bp = _find("cifar_baseline.pt")
assert _bp, "cifar_baseline.pt not found. Attach it as a Kaggle dataset, or run C1."
print("baseline:", _bp)
_ck = torch.load(_bp, map_location=DEVICE, weights_only=False)
gstate, victims = _ck["gstate"], _ck["victims"]

_rp = _find("results_cifar.json")
if _rp:
    with open(_rp) as f:
        RESULTS = json.load(f)["results"]
    print("results:", _rp, "| keys:", list(RESULTS.keys()))

train, test = load_data(cfg)
VX = torch.stack([test[i][0] for i in victims])
VY = torch.tensor([int(test[i][1]) for i in victims])
print("victims:", victims, "| labels:", VY.tolist())
print("acc check:", evaluate(gstate, DataLoader(test, batch_size=256), DEVICE),
      " (expect ~0.46)")

for _f in ("cifar_baseline.pt", "results_cifar.json", "cifar_m_partial.pt"):
    _s = _find(_f)
    if _s and not _s.startswith(cfg.out_dir):
        shutil.copy(_s, os.path.join(cfg.out_dir, _f)); print("copied", _f)


In [ ]:
# =============================================================================
# CELL C2 - NULL BASELINE.  CPU, under a minute.  Run BEFORE the attacks.
# What do trivial non-attacks score against these victims? An attack that does not
# clear these numbers has not demonstrated reconstruction, whatever it looks like.
# On PathMNIST the dataset mean beat both attacks. CIFAR is expected to differ; that
# contrast is the point.
# =============================================================================
assert not SMOKE or ALLOW_SMOKE_CELLS, "SMOKE is still True. Fix Cell 0 first."

_ri = np.random.default_rng(0).choice(len(test), size=2000, replace=False)
MEAN_IMG = torch.stack([test[int(i)][0] for i in _ri]).mean(0)
_gray = torch.full_like(VX[0], 0.5)
NULLS = {
    "constant 0.5":  lambda k: _gray,
    "dataset mean":  lambda k: MEAN_IMG,
    "uniform noise": lambda k: torch.rand(VX[0].shape,
                                          generator=torch.Generator().manual_seed(1000 + k)),
    "other victim":  lambda k: VX[(k + 5) % len(VX)],
}
null_rows = {}
print(f"{'null baseline':<16}{'PSNR':>18}{'SSIM':>18}")
for _nm, _fn in NULLS.items():
    _ps, _ss = zip(*[compute_metrics(_fn(k), VX[k]) for k in range(len(VX))])
    _pm, _ph = mean_ci(_ps); _sm, _sh = mean_ci(_ss)
    null_rows[_nm] = {"psnr_mean": _pm, "psnr_ci": _ph, "ssim_mean": _sm, "ssim_ci": _sh,
                      "psnr_all": list(_ps), "ssim_all": list(_ss)}
    print(f"{_nm:<16}{fmt(_pm, _ph):>18}{fmt(_sm, _sh):>18}")

VICTIM_FLOOR = [max(null_rows[s]["ssim_all"][k] for s in
                    ("constant 0.5", "dataset mean", "other victim")) for k in range(len(VX))]
print(f"\n{'victim':<8}{'class':<8}{'floor':>8}")
for k in range(len(VX)):
    print(f"{k:<8}{int(VY[k]):<8}{VICTIM_FLOOR[k]:>8.3f}")
_deg = [k for k in range(len(VX)) if VICTIM_FLOOR[k] > 0.6]
print(f"\nfloor: min {min(VICTIM_FLOOR):.3f}  median {np.median(VICTIM_FLOOR):.3f}  "
      f"max {max(VICTIM_FLOOR):.3f}  mean {np.mean(VICTIM_FLOOR):.3f}")
print(f"degenerate victims (floor > 0.6, threshold fixed a priori): {_deg}")
print("PathMNIST for reference: mean floor 0.411, two background tiles at 0.85 / 0.90.")
print("Thesis Table 4.4 undefended DLG on CIFAR was 0.784.")

RESULTS["null_baseline"] = {"n": len(VX), "rows": null_rows,
                            "victim_floor": VICTIM_FLOOR, "degenerate_victims": _deg,
                            "ssim_floor": float(np.mean(VICTIM_FLOOR))}
save_results("C2")


In [ ]:
# =============================================================================
# CELL C3 - ATTACKS.  RESUMABLE.  Checkpoints after every (defense, attack) pair.
#
# RUN_DEFENSES = False  -> undefended DLG + IG only, ~18 min. This is the control the
#                          PathMNIST identifiability finding needs.
# RUN_DEFENSES = True   -> the full 6 x 2 matrix, ~80 min. Gives CIFAR the same
#                          per-victim data PathMNIST has, so Table 4.4 can be restated
#                          with n=10, 95% CIs and seeded initialization.
# Re-running this cell resumes from the checkpoint; nothing is recomputed.
# =============================================================================
assert not SMOKE or ALLOW_SMOKE_CELLS, "SMOKE is still True. Fix Cell 0 first."
RUN_DEFENSES = False

_use = DEFENSES if RUN_DEFENSES else [("none", {})]
CKPT = os.path.join(cfg.out_dir, "cifar_m_partial.pt")
if os.path.exists(CKPT):
    _st = torch.load(CKPT, map_location="cpu", weights_only=False)
    defense_rows = _st["defense_rows"]
    all_recon = {tuple(k.split("|")): list(v) for k, v in _st["all_recon"].items()}
    print(f"resuming: {len(defense_rows)} conditions done ->", sorted(defense_rows))
else:
    defense_rows, all_recon = {}, {}
    print("starting from scratch")

_t0 = time.time()
for di, (dname, spec) in enumerate(_use):
    for aname, afn in ATTACKS.items():
        key = f"{dname}|{aname}"
        if key in defense_rows:
            continue
        psnrs, ssims, recs = [], [], []
        for k in range(len(VY)):
            x1, y1 = VX[k].unsqueeze(0), VY[k].view(1)
            g = capture_gradient(gstate, x1, y1, DEVICE)
            set_seed(cfg.seed + 1_000_003 * k + 97 * di)     # only the noise draw
            g = apply_defense(g, spec)
            rec, _ = afn(gstate, g, int(VY[k]), DEVICE, seed_off=k)
            ps, ss = compute_metrics(rec, VX[k])
            psnrs.append(ps); ssims.append(ss); recs.append(rec.detach().cpu())
        pm, ph = mean_ci(psnrs); sm, sh = mean_ci(ssims)
        defense_rows[key] = {"defense": dname, "attack": aname, "n": len(ssims),
                             "psnr_mean": pm, "psnr_ci": ph, "ssim_mean": sm,
                             "ssim_ci": sh, "psnr_all": psnrs, "ssim_all": ssims}
        all_recon[(dname, aname)] = recs
        torch.save({"defense_rows": defense_rows,
                    "all_recon": {f"{d}|{a}": torch.stack([r.squeeze(0) for r in v])
                                  for (d, a), v in all_recon.items()}}, CKPT)
        RESULTS["defense_matrix"] = defense_rows
        save_results(key)
        print(f"{dname:<14} {aname:<4} PSNR {fmt(pm, ph)}  SSIM {fmt(sm, sh)}  "
              f"({time.time() - _t0:.0f}s)  [checkpointed]")

torch.save({"all_recon": {f"{d}|{a}": torch.stack([r.squeeze(0) for r in v])
                          for (d, a), v in all_recon.items()},
            "VX": VX, "VY": VY, "victims": victims},
           os.path.join(cfg.out_dir, "cifar_recons.pt"))
print("\nThesis Table 4.4 for reference (n=5, std, unseeded init):")
print("  none      DLG 0.784 +/- 0.059   IG 0.476 +/- 0.053")
print("  clip C=4  DLG 0.160 +/- 0.079   IG 0.498 +/- 0.048")
print("This run uses 10 victims and seeded init, so it will NOT match exactly.")


In [ ]:
# =============================================================================
# CELL C4 - IDENTIFIABILITY + FLOOR-RELATIVE.  CPU, seconds.
# The control for the PathMNIST result. There, DLG scored LOWER on SSIM (0.145 vs
# 0.246) yet identified 7/10 victims to IG's 2/10 - the attack that looked worse
# leaked more identity. Does the same inversion appear on CIFAR, where the attacks
# genuinely reconstruct, or is it specific to low-variance imagery?
#   mean rank  - chance (n+1)/2; 1.0 is perfect identification
#   same-class - controls for the label being handed to the attack. A reconstruction
#                capturing only class-typical texture scores ~half here even when its
#                overall rank looks excellent.
# =============================================================================
if "all_recon" not in globals() or not all_recon:
    _d = torch.load(os.path.join(cfg.out_dir, "cifar_recons.pt"),
                    map_location="cpu", weights_only=False)
    all_recon = {tuple(k.split("|")): list(v) for k, v in _d["all_recon"].items()}

FL = RESULTS["null_baseline"]["victim_floor"]
DEG = set(RESULTS["null_baseline"].get("degenerate_victims", []))
keep = [k for k in range(len(VY)) if k not in DEG]
n_id = len(VY); lab = VY.tolist()
same_pairs = [(i, j) for i in range(n_id) for j in range(n_id)
              if i != j and lab[i] == lab[j]]

print(f"chance: mean rank {(n_id + 1) / 2:.2f}  top-1 1/{n_id}  "
      f"same-class {len(same_pairs) / 2:.1f}/{len(same_pairs)}\n")
print(f"{'condition':<18}{'matched':>16}{'mismatched':>16}{'gap':>16}"
      f"{'rank':>7}{'top1':>7}{'samecls':>9}{'beat':>7}{'excess':>18}")

S_ALL, ident_rows = {}, {}
for (_dn, _an), _recs in all_recon.items():
    R = [r if r.dim() == 3 else r.squeeze(0) for r in _recs]
    S = np.array([[compute_metrics(R[i], VX[j])[1] for j in range(n_id)]
                  for i in range(n_id)])
    key = f"{_dn}|{_an}"; S_ALL[key] = S
    matched = [float(S[i, i]) for i in range(n_id)]
    mism = [float(S[i, j]) for i in range(n_id) for j in range(n_id) if i != j]
    gap = [float(S[i, i] - np.delete(S[i], i).mean()) for i in range(n_id)]
    ranks = [1 + int((S[i] > S[i, i]).sum()) for i in range(n_id)]
    top1 = int(sum(r == 1 for r in ranks))
    scw = int(sum(bool(S[i, i] > S[i, j]) for i, j in same_pairs)) if same_pairs else -1
    beat = int(sum(1 for k in range(n_id) if matched[k] > FL[k]))
    exm, exh = mean_ci([matched[k] - FL[k] for k in range(n_id)])
    mm, mh = mean_ci(matched); xm, xh = mean_ci(mism); gm, gh = mean_ci(gap)
    ident_rows[key] = {"defense": _dn, "attack": _an, "n": n_id,
                       "matched": [mm, mh], "mismatched": [xm, xh], "gap": [gm, gh],
                       "ratio": float(mm / max(xm, 1e-9)), "ranks": ranks,
                       "mean_rank": float(np.mean(ranks)), "top1": top1,
                       "same_class_wins": scw, "same_class_n": len(same_pairs),
                       "beat_floor": beat, "excess_vs_floor": [exm, exh],
                       "ssim_keep_mean": float(np.mean([matched[k] for k in keep]))}
    print(f"{key:<18}{fmt(mm, mh):>16}{fmt(xm, xh):>16}{fmt(gm, gh):>16}"
          f"{np.mean(ranks):>7.2f}{f'{top1}/{n_id}':>7}"
          f"{(f'{scw}/{len(same_pairs)}' if same_pairs else 'n/a'):>9}"
          f"{f'{beat}/{n_id}':>7}{fmt(exm, exh):>18}")

print("\nmatched / mismatched ratio - is the leakage victim-specific or generic?")
for k, r in ident_rows.items():
    print(f"  {k:<18}{r['ratio']:>6.2f}x")
print("  PathMNIST undefended was DLG 4.71x, IG 1.62x")

RESULTS["identifiability"] = ident_rows
save_results("C4")
np.savez(os.path.join(cfg.out_dir, "cifar_cross_ssim.npz"),
         **{k.replace("|", "__"): v for k, v in S_ALL.items()})
print("[saved]", os.path.join(cfg.out_dir, "cifar_cross_ssim.npz"))


In [ ]:
# =============================================================================
# CELL C5 - SUMMARY.  Side-by-side with PathMNIST and with thesis Table 4.4.
# Table 4.4 reports mean +/- std over 5 inputs; everything here is a 95% CI over 10
# victims. Converting for n=5: half-width = t(0.975,4)*std/sqrt(5) = 1.2415*std.
# =============================================================================
_T5 = 1.2415
TAB44 = {"none": (0.784, 0.059, 0.476, 0.053), "clip C=4": (0.160, 0.079, 0.498, 0.048),
         "noise s=0.01": (0.146, 0.034, 0.461, 0.039),
         "noise s=0.1": (0.060, 0.020, 0.161, 0.052),
         "prune 90%": (0.096, 0.024, 0.252, 0.049),
         "combined": (0.090, 0.031, 0.243, 0.050)}
PATH = {"acc": 0.6077, "floor": 0.411,
        "none": (0.145, 0.073, 0.246, 0.163), "clip C=4": (0.058, 0.056, 0.246, 0.163),
        "noise s=0.01": (0.023, 0.018, 0.204, 0.123),
        "noise s=0.1": (0.030, 0.018, 0.100, 0.048),
        "prune 90%": (0.047, 0.033, 0.132, 0.063),
        "combined": (0.042, 0.026, 0.135, 0.059)}

print("=" * 78)
print("CIFAR-10 control study")
print("=" * 78)
b = RESULTS["baseline"]
print(f"\nbaseline accuracy  this run {b['final_acc']:.4f}   original notebook 0.460"
      f"   PathMNIST {PATH['acc']:.4f}")
nb_ = RESULTS["null_baseline"]
print(f"null SSIM floor    CIFAR {nb_['ssim_floor']:.3f}   PathMNIST {PATH['floor']:.3f}")

print(f"\n{'defense':<14}{'DLG this run':>20}{'DLG Tab4.4':>18}"
      f"{'IG this run':>20}{'IG Tab4.4':>18}")
for d, (dm, ds, im, isd) in TAB44.items():
    r = RESULTS.get("defense_matrix", {})
    g = lambda a: r.get(f"{d}|{a}")
    f_ = lambda x: fmt(x["ssim_mean"], x["ssim_ci"]) if x else "not run"
    print(f"{d:<14}{f_(g('DLG')):>20}{f'{dm:.3f} +/- {ds * _T5:.3f}':>18}"
          f"{f_(g('IG')):>20}{f'{im:.3f} +/- {isd * _T5:.3f}':>18}")

if "identifiability" in RESULTS:
    print(f"\n{'IDENTIFIABILITY':<18}{'rank':>7}{'top1':>7}{'samecls':>9}"
          f"{'ratio':>8}{'beat floor':>12}")
    for k, r in RESULTS["identifiability"].items():
        t1 = f"{r['top1']}/{r['n']}"
        sc = f"{r['same_class_wins']}/{r['same_class_n']}"
        bf = f"{r['beat_floor']}/{r['n']}"
        print(f"{k:<18}{r['mean_rank']:>7.2f}{t1:>7}{sc:>9}"
              f"{r['ratio']:>8.2f}{bf:>12}")
    print(f"\nPathMNIST undefended: DLG rank 2.20 top1 7/10 ratio 4.71x beat 3/10")
    print(f"                      IG  rank 3.70 top1 2/10 ratio 1.62x beat 4/10")
    print("\nThe question this answers: on PathMNIST the attack with the LOWER SSIM")
    print("identified far more victims. If CIFAR repeats that ordering the finding")
    print("generalises; if not, it is specific to low-variance imagery. Either way it")
    print("belongs in the paper - report what came out, not the nicer one.")

save_results("C5")
print(f"\nEverything is in {cfg.out_dir} - download it before closing the session.")
